https://confident-reflection-production-d304.up.railway.app/api/admin/extract-all?token=CHOCOLATCOGNITIF


### Extraction of the data and DataFrame construction

In [72]:
from pathlib import Path
import json
import pandas as pd

rows = []

for file in Path("archives").rglob("*.json"):
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)

    for trial in data["resultats"]:
        if trial.get("task") != "target":
            continue

        participant_id = trial["participant_id"]
        rt = trial["rt"]
        if not rt:
            continue

        valence = trial["valence"]
        attr = trial["baseline_attrac"]
        response = trial["response_meaning"]

        # Congruence
        if (valence == "positif" and attr == "Attractif") or \
        (valence == "negatif" and attr == "Unattractif"):
            congruence = "congruent"
        else:
            congruence = "incongruent"
        if valence=='neutre':
            congruence='neutre'

        # Résultat
        if attr == "Attractif" and response == "Attractif":
            result = 1
        elif attr == "Unattractif" and response == "Non attractif":
            result = 1
        else:
            result = 0

        rows.append({
            "id": participant_id,
            "valence_mot":valence,
            "actracttiveness_face":attr, 
            "congruence": congruence,
            "correct": result,
            "rt": rt/1000
        })

# Transformer en DataFrame
df = pd.DataFrame(rows)

# Export en CSV
df.to_csv("resultats.csv", index=False, encoding="utf-8")

print("Fichier CSV généré")
df

Fichier CSV généré


,id,valence_mot,actracttiveness_face,congruence,correct,rt
0,9at0mps0,positif,Attractif,congruent,0,0.223
1,9at0mps0,positif,Unattractif,incongruent,0,0.295
2,9at0mps0,negatif,Unattractif,congruent,1,0.109
3,9at0mps0,neutre,Unattractif,neutre,0,0.185
4,9at0mps0,positif,Unattractif,incongruent,0,1.067
...,...,...,...,...,...,...
5673,6ipwu4mk,neutre,Unattractif,neutre,1,0.620
5674,6ipwu4mk,negatif,Attractif,incongruent,0,0.879
5675,6ipwu4mk,negatif,Attractif,incongruent,1,0.546
5676,6ipwu4mk,neutre,Attractif,neutre,1,0.660


### Detect the outliers

Visage précédé d'un mot neutre. En théorie, 100% de précision:

In [73]:
# We take all the rows where the congruence condition is neutral
control = df[df['congruence'] == 'neutre']
# Pourcentage of correct answer per id
control_pourcentage = control.groupby('id')['correct'].mean()
# Get the index of the id that got a pourcentage of correct answer below 70%
id_outliers = control_pourcentage[control_pourcentage < 0.70].index

### Condition neutral word

In [74]:
# We take all the rows where the valence of the word is neutral
neutral = df[df['valence_mot'] == 'neutre']
# Construct a new datafram without outliers
data_without_outliers_neutral= neutral[~neutral['id'].isin(id_outliers)]
# Export en CSV
data_without_outliers_neutral.to_csv("data_without_outliers_neutral.csv", index=False, encoding="utf-8")
print("Fichier CSV généré")

Fichier CSV généré


In [75]:
# Mean pourcentage of correct answer
data_without_outliers_neutral['correct'].mean()

np.float64(0.8551617873651772)

### Condition positive word

In [76]:
# We take all the rows where the valence of the word is positive
positive = df[df['valence_mot'] == 'positif']
# Construct a new datafram without outliers
data_without_outliers_positive= positive[~positive['id'].isin(id_outliers)]
# Export en CSV
data_without_outliers_positive.to_csv("data_without_outliers_positive.csv", index=False, encoding="utf-8")
print("Fichier CSV généré")

Fichier CSV généré


In [77]:
# Mean pourcentage of correct answer
data_without_outliers_positive['correct'].mean()

np.float64(0.8176881303335919)

### Condition negative word


In [78]:
# We take all the rows where the valence of the word is positive
negative = df[df['valence_mot'] == 'negatif']
# Construct a new datafram without outliers
data_without_outliers_negative= negative[~negative['id'].isin(id_outliers)]
# Export en CSV
data_without_outliers_negative.to_csv("data_without_outliers_negative.csv", index=False, encoding="utf-8")
print("Fichier CSV généré")

Fichier CSV généré


In [79]:
# Mean pourcentage of correct answer
data_without_outliers_negative['correct'].mean()

np.float64(0.83203125)

### Resultat condition congruente

In [80]:
print(data_without_outliers_positive[data_without_outliers_positive['congruence']=='congruent']['correct'].mean())
print(data_without_outliers_negative[data_without_outliers_negative['congruence']=='congruent']['correct'].mean())

0.694488188976378
0.9347826086956522


### Resultat condition incongruente

In [81]:
print(data_without_outliers_positive[data_without_outliers_positive['congruence']=='incongruent']['correct'].mean())
print(data_without_outliers_negative[data_without_outliers_negative['congruence']=='incongruent']['correct'].mean())

0.9373088685015291
0.7279874213836478
